### Pipeline ML afin de vérifier les résultats du papier : Cascading Machine Learning to Attack Bitcoin Anonymity

Bitcoin est une cryptomonnaie décentralisée et pseudonyme. Bien que les transactions soient publiques, l'identité des utilisateurs est protégée par l'anonymat, ce qui a favorisé son utilisation pour des activités illicites (blanchiment, darknet). Ce projet implémente la méthode de Zola et al. (2019) pour caractériser les entités du réseau en utilisant uniquement des données extraites de la blockchain.

#### Importation des modules nécessaires

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

# Importation des fonctions depuis utils.py
from utils import evaluate_classifier, prepare_and_init_model, run_cascade_layer
from data_gen import generate_entity_df, generate_address_df, generate_motif1_df, generate_motif2_df

print("Modules chargés avec succès.")

Modules chargés avec succès.


#### Chargement des données  
En l'absence des données réelles au début du TER, nous avons généré des données synthétiques (Mock Data) pour valider le pipeline ML.  
Première cellule : Données réelles collectés dans features_extraction.ipynb.  
Deuxième cellule : Mock data respectant les features et proportions du papier.  
Troisième cellule : Mock data relativement complexe pour pouvoir observer une différence entre la baseline et le final. Sur les petites données, les scores étaient identiques et laissaient penser à une erreur dans la logique du code.   
Quatrième cellule : Mock data initiales pour tester le bon fonctionnement du pipeline. On peut décommenter les lignes concernant motif2.

In [2]:
in_dir = "data_processed/2024_all"

# Importation des CSV
df_entity = pd.read_csv(f"{in_dir}/df_entity.csv")
df_address = pd.read_csv(f"{in_dir}/df_address.csv")
df_motif1 = pd.read_csv(f"{in_dir}/df_motif1.csv")
df_motif2 = pd.read_csv(f"{in_dir}/df_motif2.csv")

# Ajout de la vérité terrain aux niveaux inférieurs
df_address = df_address.merge(df_entity[['entity_id', 'label']], on='entity_id', how='left')
df_motif1 = df_motif1.merge(df_entity[['entity_id', 'label']], on='entity_id', how='left')
df_motif2 = df_motif2.merge(df_entity[['entity_id', 'label']], on='entity_id', how='left')

# Nettoyage d'éventuels NaN générés lors de l'extraction
df_address = df_address.fillna(0)
df_motif1 = df_motif1.fillna(0)
df_motif2 = df_motif2.fillna(0)

# Définition des colonnes à ignorer par les modèles
ignore_cols_entity = ['entity_id', 'label']
ignore_cols_add = ['entity_id', 'address_id', 'label']
ignore_cols_m1 = ['entity_id', 'motif_id', 'label', 'entity_id_recv', 'tx_hash']
ignore_cols_m2 = ['entity_id', 'label','motif_id_1', 'motif_id_2','entity_id_2', 'entity_id_3']

print("Données réelles chargées et prêtes pour la cascade :")
print(f" - {len(df_entity)} Entités")
print(f" - {len(df_address)} Adresses")
print(f" - {len(df_motif1)} 1-Motifs")
print(f" - {len(df_motif2)} 2-Motifs")

Données réelles chargées et prêtes pour la cascade :
 - 73 Entités
 - 39902 Adresses
 - 1408 1-Motifs
 - 960 2-Motifs


In [3]:
%%script false --no-raise-error

# Génération de données réalistes
df_entity = generate_entity_df(n=350)
ignore_cols_entity = ['entity_id', 'entity_name']

df_address = generate_address_df(df_entity, sampling_ratio=0.1)
ignore_cols_add = ['entity_id', 'address_id', 'label']

df_motif1 = generate_motif1_df(df_entity,sampling_ratio=0.2)
ignore_cols_m1 = ['entity_id', 'motif_id', 'label']

#df_motif2 = generate_motif2_df(df_entity)
#ignore_cols_m2 = ['entity_id', 'motif_id', 'label']

In [4]:
%%script false --no-raise-error
# --- GÉNÉRATION DE DONNÉES SEMI-RÉALISTES (LOGIQUE "PIÈGE") ---

# Configuration
n_entities = 1000
classes = ['Exchange', 'Mixer', 'Gambling', 'Marketplace']
np.random.seed(42) 

# 1. GÉNÉRATION DES ENTITÉS (Le "Flou" pour la Baseline)
# Stratégie : On rend le 'total_balance' trompeur. 
# Normalement un Exchange est riche et un Mixer est moyen.
# Ici, on va mélanger les distributions pour que la Baseline hésite.

data_ent = []
for i in range(n_entities):
    label = np.random.choice(classes)
    
    # Logique de solde ambiguë
    if label == 'Exchange':
        # Majorité riches, mais certains pauvres (piège)
        balance = np.random.uniform(1000, 10000) if np.random.rand() > 0.2 else np.random.uniform(50, 200)
        n_tx = np.random.randint(1000, 5000)
    elif label == 'Mixer':
        # Beaucoup ressemblent à du Gambling (solde moyen)
        balance = np.random.uniform(100, 1000)
        n_tx = np.random.randint(500, 2000)
    elif label == 'Gambling':
        balance = np.random.uniform(50, 500)
        n_tx = np.random.randint(100, 1000)
    else: # Marketplace
        balance = np.random.uniform(10, 300)
        n_tx = np.random.randint(10, 200)
        
    data_ent.append([i, f"Ent_{i}", balance, np.random.randint(5, 50), n_tx, label])

df_entity = pd.DataFrame(data_ent, columns=['entity_id', 'entity_name', 'total_balance', 'n_addresses', 'n_transactions', 'label'])
ignore_cols_entity = ['entity_id', 'entity_name']

print("Répartition des classes :")
print(df_entity['label'].value_counts())


# 2. GÉNÉRATION DES ADRESSES (La "Vérité" pour la Cascade)
# Ici, on met des patterns très clairs. Si le modèle regarde l'adresse, il doit deviner la classe à 100%.

data_add = []
for _, row in df_entity.iterrows():
    # Nombre d'adresses par entité (variable)
    n_addr = np.random.randint(5, 25) 
    
    for _ in range(n_addr):
        # CARACTÉRISTIQUES DISCRIMINANTES (La signature comportementale)
        
        # SIBLINGS (Frères) : Les Mixers créent des clusters énormes
        if row['label'] == 'Mixer':
            n_siblings = np.random.randint(20, 50) # Signature forte
            is_unique = 1 # Ne réutilise jamais
        elif row['label'] == 'Exchange':
            n_siblings = np.random.randint(0, 5)   # Peu de clustering direct
            is_unique = 0 # Réutilise souvent les adresses de dépôt
        else:
            n_siblings = np.random.randint(0, 3)
            is_unique = np.random.randint(0, 2)
            
        # BALANCE ADRESSE : Les Gambling sites ont des montants fixes/ronds (simulé ici par std faible)
        if row['label'] == 'Gambling':
            addr_bal = np.random.normal(50, 5) 
        else:
            addr_bal = np.random.exponential(50)

        data_add.append([
            f"A_{np.random.randint(1e9)}", 
            row['entity_id'], 
            addr_bal, 
            np.random.rand()*100, 
            is_unique, 
            n_siblings,
            row['label'] # Nécessaire pour le split stratifié
        ])

df_address = pd.DataFrame(data_add, columns=['address_id', 'entity_id', 'addr_recv_amount', 'addr_balance', 'is_unique', 'n_siblings', 'label'])
ignore_cols_add = ['entity_id', 'address_id', 'label']


# 3. GÉNÉRATION DES MOTIFS (Bonus de précision)
# On simule un "Motif 1" qui détecte les boucles (typique des Mixers et Marketplaces)

data_m1 = []
for _, row in df_entity.iterrows():
    # Signature Motif : Force du cycle
    if row['label'] in ['Mixer', 'Marketplace']:
        cycle_strength = np.random.uniform(0.7, 1.0) # Fort
    else:
        cycle_strength = np.random.uniform(0.0, 0.4) # Faible
        
    data_m1.append([f"M1_{row['entity_id']}", row['entity_id'], cycle_strength, row['label']])

df_motif1 = pd.DataFrame(data_m1, columns=['motif_id', 'entity_id', 'cycle_strength', 'label'])
ignore_cols_m1 = ['entity_id', 'motif_id', 'label']

# Pas de Motif 2 pour l'instant pour alléger
df_motif2 = pd.DataFrame(columns=['entity_id', 'label']) # Vide mais existant pour ne pas casser le code

print(f"\nDonnées générées : {len(df_entity)} Entités, {len(df_address)} Adresses.")
print("Scénario : Les 'statistiques globales' sont floues, mais les 'comportements locaux' (adresses) sont clairs.")

#### Mock Data originelles  

In [5]:
%%script false --no-raise-error

# Création du df_entity (inventé car pas de données pour l'instant)
data_entity = {
    'entity_id': [101, 102, 103, 104, 105],
    'entity_name': ['Kraken', 'BitMixer', 'SatoshiDice', 'Binance', 'SilkRoad'],
    'total_balance': [5000.5, 120.0, 300.5, 8000.0, 50.0],
    'n_addresses': [100, 20, 500, 150, 10],             
    'n_transactions': [5000, 400, 12000, 7000, 50],      
    'label': ['Exchange', 'Mixer', 'Gambling', 'Exchange', 'Marketplace']
}
df_entity = pd.DataFrame(data_entity)

# Stockages des variables inutiles de df_entity (et sa variante df_final) pour l'entraînement des modèles
ignore_cols_entity = ['entity_id', 'entity_name']

# Création du df_address (idem)
data_address = {
    'address_id': [
        'A1', 'A2', 'A3', 'A4',       # Kraken (Exchange)
        'B1', 'B2', 'B3',             # BitMixer (Mixer)
        'C1', 'C2', 'C3',             # SatoshiDice (Gambling) 
        'D1', 'D2', 'D3', 'D4',       # Binance (Exchange)
        'E1', 'E2'                    # SilkRoad (Marketplace)
    ],
    'addr_recv_amount': np.random.rand(16) * 1000, 
    'addr_balance': np.random.rand(16) * 100,
    'is_unique': [0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0],
    'n_siblings': np.random.randint(0, 20, 16),
    'entity_id': [
        101, 101, 101, 101,
        102, 102, 102,
        103, 103, 103,
        104, 104, 104, 104,
        105, 105
    ]
}
df_address = pd.DataFrame(data_address)

# Attribution à chaque adresse du label de son entité
df_address = df_address.merge(df_entity[['entity_id', 'label']], on='entity_id', how='left')
# Stockage des variables inutiles de df_address pour l'entraînement des modèles
ignore_cols_add = ['entity_id', 'address_id']

# Création du df_motif1 (idem)
data_motif1 = {
    'motif_id': [f'M1_{i}' for i in range(15)],
    'entity_id': [101, 101, 101, 102, 102, 103, 103, 103, 104, 104, 104, 104, 105, 105, 105],
    'motif_weight': np.random.rand(15),       
    'motif_degree': np.random.randint(1, 5, 15)
}
df_motif1 = pd.DataFrame(data_motif1)

# Attribution à chaque 1_motif du label de son entité
df_motif1 = df_motif1.merge(df_entity[['entity_id', 'label']], on='entity_id', how='left')
# Stockages des variables inutiles de df_motif1 pour l'entraînement des modèles
ignore_cols_m1 = ['entity_id', 'motif_id']

# Création du df_motif2 (idem)
data_motif2 = {
    'motif_id': [f'M2_{i}' for i in range(12)], 
    'entity_id': [101, 101, 102, 102, 103, 103, 103, 104, 104, 105, 105, 101],
    'closeness_centrality': np.random.rand(12), 
    'pattern_size': np.random.randint(3, 10, 12) 
}
df_motif2 = pd.DataFrame(data_motif2)
# Attribution à chaque 2_motif du label de son entité
df_motif2 = df_motif2.merge(df_entity[['entity_id', 'label']], on='entity_id', how='left')
# Stockages des variables inutiles de df_motif2 pour l'entraînement des modèles
ignore_cols_m2 = ['entity_id', 'motif_id']

print("Mock data chargées : Entité, Address, Motif1, Motif2.")

#### Entraînement du classifieur de référence (Baseline) C_entity​ utilisant uniquement les attributs globaux des entités, sans l'enrichissement par cascade.  


In [6]:
# Entraînement d'un classifier sur le df_entity brut pour servir de comparaison

MODEL_TYPE = "GradientBoosting"
# N_SPLITS = 5 dans le papier
N_SPLITS = 2
# Mettre False pour performance
RETURN_DETAILED_REPORT=True

# Initialisation du Classifier (paramètres issus du papier)
C_entity, X_ent, y_ent = prepare_and_init_model(df_entity, ignore_cols_entity, model_type=MODEL_TYPE)

# Evaluation du classifier C_entity par Stratified K-Fold Cross Validation
res_baseline = evaluate_classifier(C_entity, X_ent, y_ent, n_splits=N_SPLITS, return_detailed_report=RETURN_DETAILED_REPORT)
baseline_acc = res_baseline["acc_mean"]
baseline_acc_std = res_baseline["acc_std"]
baseline_mcc = res_baseline["mcc_mean"]
baseline_mcc_std = res_baseline["mcc_std"]


Préparation du modèle : GradientBoosting
Features sélectionnées : ['amount_received', 'amount_sent', 'balance', 'n_tx_received', 'n_tx_sent', 'n_addr_received_from', 'n_addr_sent_to']
Accuracy moyenne : 0.45 (+/- 0.02)
MCC moyenne : 0.03 (+/- 0.05)


#### Entraînement des classifiers C_address, C_motif1 et C_motif2, prédiction grâce à ces derniers des classes des adresses, 1_motifs et 2_motifs séléctionnés pour la génération des nouvelles features, création des nouvelles features pour enrichir df_entity.

In [7]:
MODEL_TYPE = 'RandomForest'
TEST_SIZE = 0.3

new_features_add = run_cascade_layer(df_address, 'add', ignore_cols_add, model_type=MODEL_TYPE, test_size=TEST_SIZE)
new_features_motif1 = run_cascade_layer(df_motif1, 'mot1', ignore_cols_m1, model_type=MODEL_TYPE, test_size=TEST_SIZE)
new_features_motif2 = run_cascade_layer(df_motif2, 'mot2', ignore_cols_m2, model_type=MODEL_TYPE, test_size=TEST_SIZE)

ValueError: The least populated classes in y have only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2. Classes with too few members are: ['old-historic']

#### Phase de Cascade : Fusion des features avec df_entity pour entraîner le modèle final C_final​.

In [21]:
df_final = df_entity.copy()

# Enrichissement de df_entity avec les nouvelles features calculées
df_final = df_final.merge(new_features_add, on='entity_id', how='left')
df_final = df_final.merge(new_features_motif1, on='entity_id', how='left')
#df_final = df_final.merge(new_features_motif2, on='entity_id', how='left')

# Si une entité n'avait aucune adresse dans le Set B (improbable mais peut arrivé sur faux jeu de données),
# Remplacement de tous les nan par 0.0 
cascade_cols = [c for c in df_final.columns if ' as ' in c]
df_final[cascade_cols] = df_final[cascade_cols].fillna(0.0)

MODEL_TYPE = 'GradientBoosting'
# N_SPLITS = 5 dans le papier
N_SPLITS = 5
# Mettre False pour performance
RETURN_DETAILED_REPORT=True

# Initialisation du Classifier (paramètres issus du papier)
C_final, X_final, y_final = prepare_and_init_model(df_final, ignore_cols_entity, model_type=MODEL_TYPE)

# Evaluation du classifier C_final par Stratified K-Fold Cross Validation
res_final = evaluate_classifier(C_final, X_final, y_final, n_splits=N_SPLITS, return_detailed_report=RETURN_DETAILED_REPORT)
final_acc = res_final["acc_mean"]
final_acc_std = res_final["acc_std"]
final_mcc = res_final["mcc_mean"]
final_mcc_std = res_final["mcc_std"]

Préparation du modèle : GradientBoosting
Features sélectionnées : ['amount_received', 'amount_sent', 'balance', 'n_tx_received', 'n_tx_sent', 'n_addr_received_from', 'n_addr_sent_to', 'add as exchanges', 'add as gambling', 'add as services-others', 'mot1 as exchanges', 'mot1 as services-others']


/home/dev/Code_TER/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Accuracy moyenne : 0.55 (+/- 0.09)
MCC moyenne : 0.13 (+/- 0.16)


/home/dev/Code_TER/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/dev/Code_TER/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/dev/Code_TER/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/dev/Code_TER/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precis

#### Résultats

In [22]:
res_data = {
    'Mean Accuracy': [baseline_acc, final_acc],
    'Std Deviation': [baseline_acc_std, final_mcc_std],
    'MCC' : [baseline_mcc, final_mcc]
}
df_results = pd.DataFrame(res_data, index=['Baseline', 'Final (Cascade)'])

print("\n Comparaison Baseline / Final")
display(df_results)

# Extraction des différentes metrics de 'detailed_report'
rows = []

# Récupération de la liste des classes (en excluant les moyennes globales)
classes = [k for k in res_baseline['detailed_report'].keys() if k not in ['accuracy', 'macro avg', 'weighted avg']]

for cls in classes:
    # Récupération Baseline
    base_prec = res_baseline['detailed_report'][cls]['precision']
    base_rec  = res_baseline['detailed_report'][cls]['recall']
    base_f1   = res_baseline['detailed_report'][cls]['f1-score']
    
    # Récupération Final
    fin_prec = res_final['detailed_report'][cls]['precision']
    fin_rec  = res_final['detailed_report'][cls]['recall']
    fin_f1   = res_final['detailed_report'][cls]['f1-score']
    
    rows.append([cls, base_prec, fin_prec, base_rec, fin_rec, base_f1, fin_f1])

# Création du DataFrame détaillant les résultats
df_details = pd.DataFrame(rows, columns=[
    'Classe', 
    'Prec. (Base)', 'Prec. (Final)', 
    'Recall (Base)', 'Recall (Final)', 
    'F1 (Base)', 'F1 (Final)'
])

print("\n Différentes metrics par classe")
# Affichage avec format pourcentage
format_dict = {col: "{:.2%}" for col in df_details.columns if col != 'Classe'}
display(df_details.style.format(format_dict))


 Comparaison Baseline / Final


,Mean Accuracy,Std Deviation,MCC
Baseline,0.534159,0.006381,0.093486
Final (Cascade),0.548571,0.081925,0.130997



 Différentes metrics par classe


,Classe,Prec. (Base),Prec. (Final),Recall (Base),Recall (Final),F1 (Base),F1 (Final)
0,exchanges,57.63%,58.33%,85.00%,87.50%,68.69%,70.00%
1,gambling,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
2,old-historic,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
3,pools,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
4,services-others,38.46%,41.67%,26.32%,26.32%,31.25%,32.26%
